# Day 7: Week 1 Checkpoint & EDA Report Generation

Welcome to the end of Week 1! 

Today is all about documentation and verification. The Zidio guidelines strictly require an "EDA report" and proof that our baseline models are logged. 

Instead of manually writing this out, we are going to write a Python script that:
1. **Verifies our Data:** Checks the shapes and quality of our cleaned datasets.
2. **Queries MLflow:** Pulls the exact MAPE and RMSE metrics from our Prophet and LSTM runs.
3. **Generates a Report:** Automatically writes a professional `Week1_Checkpoint_Report.md` file. This will eventually be converted into the final PDF submission!

Let's close out Week 1 strong.

In [1]:
import pandas as pd
import mlflow
import os
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

print("🔍 Running Week 1 Checkpoint Verifications...\n")

# ==========================================
# 1. Data & EDA Checkpoint
# ==========================================
print("--- 1. Dataset Verification ---")
customers = pd.read_csv('data/customers_segmented.csv')
transactions = pd.read_csv('data/transactions_cleaned.csv')
daily_sales = pd.read_csv('data/daily_sales_prepared.csv')

print(f"✅ Customers Dataset: {len(customers)} rows | Segments: {customers['segment'].nunique()}")
print(f"✅ Transactions Dataset: {len(transactions)} rows | Missing values: {transactions.isnull().sum().sum()}")
print(f"✅ Time-Series Dataset: {len(daily_sales)} continuous days")

🔍 Running Week 1 Checkpoint Verifications...

--- 1. Dataset Verification ---
✅ Customers Dataset: 2000 rows | Segments: 3
✅ Transactions Dataset: 62611 rows | Missing values: 0
✅ Time-Series Dataset: 541 continuous days


### Querying MLflow for Baseline Metrics
Now we need to prove our models are tracked. We will use the `mlflow.search_runs()` function to query our local `mlruns` database and pull the exact MAPE scores we logged on Day 5 and Day 6.

In [2]:
# ==========================================
# 2. MLflow Checkpoint
# ==========================================
print("\n--- 2. MLflow Experiment Tracking ---")

# Query the local MLflow database for our specific experiment
try:
    runs = mlflow.search_runs(experiment_names=["RetailPulse_Demand_Forecasting"])
    
    if not runs.empty:
        # Display a clean table of our runs
        display_cols = ['tags.mlflow.runName', 'params.model_type', 'metrics.mape_percent', 'metrics.rmse']
        print("Logged Runs:")
        display(runs[display_cols])
        
        # Extract the specific MAPE scores for our report
        prophet_mape = runs[runs['params.model_type'] == 'Prophet']['metrics.mape_percent'].values[0]
        lstm_mape = runs[runs['params.model_type'] == 'LSTM_PyTorch']['metrics.mape_percent'].values[0]
    else:
        print("⚠️ No runs found. Ensure you completed Day 5 and Day 6.")
        prophet_mape, lstm_mape = "N/A", "N/A"
except Exception as e:
    print(f"Could not query MLflow: {e}")
    prophet_mape, lstm_mape = "N/A", "N/A"


--- 2. MLflow Experiment Tracking ---
Logged Runs:


,tags.mlflow.runName,params.model_type,metrics.mape_percent,metrics.rmse
0,Baseline_LSTM_v1,LSTM_PyTorch,18.81,8132.25
1,Baseline_Prophet_v1,Prophet,12.48,5504.44


### Generating the Week 1 EDA Report
The Zidio PDF requires a comprehensive Project Documentation (10-18 pages). We are going to start building that right now by generating a structured Markdown report. 

This report summarizes our EDA findings, data quality checks, segmentation results, and baseline model performance.

In [3]:
# ==========================================
# 3. Generate Week 1 EDA Report
# ==========================================
print("\n--- 3. Generating EDA Report ---")

# Create a reports folder
os.makedirs('reports', exist_ok=True)

# Calculate some quick stats for the report
date_min = transactions['date'].min()
date_max = transactions['date'].max()
segment_counts = customers['segment'].value_counts().to_dict()

# Build the Markdown content
report_content = f"""# RetailPulse - Week 1 Checkpoint & EDA Report
**Generated on:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Project:** AI-Powered Customer Analytics & Demand Forecasting  

---

## 1. Project Overview
RetailPulse is an end-to-end data science platform designed to ingest retail sales, customer, and inventory data to deliver accurate demand forecasts, customer segmentation, and churn predictions. 

## 2. Dataset Summary & EDA
- **Total Customers:** {len(customers):,}
- **Total Transactions:** {len(transactions):,}
- **Date Range:** {date_min} to {date_max} ({len(daily_sales)} continuous days)

**Key EDA Findings:**
- Transaction amounts are heavily right-skewed (typical retail distribution).
- Clear weekly seasonality observed (weekend spikes).
- Massive holiday seasonality observed in Nov/Dec.
- 2% missing values in `payment_method` were successfully imputed using the mode.

## 3. Data Quality & Validation
- **Great Expectations:** All 5 critical data quality rules passed (no negative amounts, unique IDs, no nulls in critical columns).
- **Stationarity:** Augmented Dickey-Fuller (ADF) test confirmed the presence of a trend (non-stationary), which is expected and handled by Prophet/LSTM.

## 4. Customer Segmentation (RFM)
Using K-Means (k=5) and DBSCAN for outlier detection, we identified the following business segments:
- **Segments:** {list(segment_counts.keys())}
- **Distribution:** {segment_counts}

## 5. Baseline Forecasting Models (Logged in MLflow)
We established two baseline models for 30-day ahead demand forecasting:
- **Prophet (Statistical):** MAPE = {prophet_mape}%
- **LSTM (Deep Learning):** MAPE = {lstm_mape}%

*Note: Both models are fully tracked in MLflow with parameters, metrics, and visual artifacts.*

## 6. Next Steps (Week 2)
- Build the **Hybrid Ensemble Model** (Prophet + LSTM) to push MAPE ≤ 12%.
- Implement **Churn Prediction** using XGBoost and SHAP explainability.
- Develop **Inventory Optimization** logic (Safety Stock, Reorder Point).
"""

# Save the report
report_path = 'reports/Week1_Checkpoint_Report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"✅ Week 1 Checkpoint Report successfully saved to: {report_path}")


--- 3. Generating EDA Report ---
✅ Week 1 Checkpoint Report successfully saved to: reports/Week1_Checkpoint_Report.md


### Wrap Up: Week 1 Complete! 🎉

You have successfully completed the first 25% of the Zidio project. 

**Your Week 1 Deliverables:**
- [x] Cleaned, validated dataset (Great Expectations).
- [x] Customer Segmentation (K-Means, DBSCAN, RFM).
- [x] Time-Series Prep (Stationarity, Decomposition).
- [x] Baseline Models (Prophet, LSTM) logged in MLflow.
- [x] Week 1 EDA Report generated.

Tomorrow, we start **Week 2**. We are going to take our Prophet and LSTM models, combine them into a **Hybrid Ensemble**, and tune it to crush that **MAPE ≤ 12%** target!